
# Chemical evolution: How SFH and outflows shape metal enrichment history

Six perspectives on chemical evolution: (1) closed-box model with varying SFR
timescales; (2) cumulative metallicity from different exponential SFHs; (3)
leaky-box model showing how outflow rates suppress Z; (4) age-metallicity
relation across galactic radii; (5) three metallicity evolution scenarios
(constant solar, linear ramp, two-step); (6) resulting integrated SEDs showing
how Z(t) pathways alter optical/UV colors and absorption features. Together
they show how star formation, galactic winds, and chemical enrichment control
the Z(t) history and observable photometry.


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style
from tengri.cosmology import age_at_z0
from tengri.sfh import closed_box_metallicity

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Time axis: look-back time in Gyr (cosmology-dependent age)
age_uni_gyr = float(age_at_z0())
t_gyr = np.linspace(0, age_uni_gyr, 200)
t_yr = t_gyr * 1e9

# Solar metallicity
Z_sun = 10.0 ** (-1.848)

# --- Panel 1: Closed-box model with different SFR timescales ---
ax = axes[0, 0]
age_from_start = age_uni_gyr - t_gyr
for tau_gyr, y_label in [(2.0, "τ=2 Gyr"), (5.0, "τ=5 Gyr"), (10.0, "τ=10 Gyr")]:
    sfr = np.exp(-age_from_start / tau_gyr)
    log_z = closed_box_metallicity(t_yr, sfr, yield_y=0.03, eta_outflow=0.0, f_gas_init=0.9)
    ax.plot(t_gyr, 10.0 ** np.array(log_z), lw=2.0, label=y_label)

ax.set_xlabel("Look-back Time [Gyr]")
ax.set_ylabel(r"Metallicity (Z / Z$_\odot$)")
ax.legend(fontsize=10, frameon=False)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 14)

# --- Panel 2: Varying SFR shapes ---
ax = axes[0, 1]
tau_values = [1.0, 2.0, 5.0, 10.0]
for tau_gyr in tau_values:
    cum_mass = 1.0 - np.exp(-t_gyr / tau_gyr)
    z_evolve = Z_sun * 0.5 * cum_mass
    ax.plot(t_gyr, z_evolve / Z_sun, lw=1.5, label=f"τ={tau_gyr:.1f} Gyr")

ax.set_xlabel("Look-back Time [Gyr]")
ax.set_ylabel(r"Metallicity (Z / Z$_\odot$)")
ax.legend(fontsize=10, frameon=False)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 14)

# --- Panel 3: Leaky-box model (outflow dependence) ---
ax = axes[1, 0]
outflow_rates = [0.0, 0.2, 0.5, 0.8]
colors = plt.cm.Reds(np.linspace(0.3, 0.9, len(outflow_rates)))

sfr_const = np.ones_like(t_yr)
for eta, color in zip(outflow_rates, colors):
    log_z = closed_box_metallicity(t_yr, sfr_const, yield_y=0.03, eta_outflow=eta, f_gas_init=0.9)
    ax.plot(t_gyr, 10.0 ** np.array(log_z), lw=1.5, color=color, label=f"η={eta:.1f}")

ax.set_xlabel("Look-back Time [Gyr]")
ax.set_ylabel(r"Metallicity (Z / Z$_\odot$)")
ax.legend(fontsize=10, frameon=False)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 14)

# --- Panel 4: Age-metallicity relation ---
ax = axes[1, 1]
ages_gyr = np.array([2.0, 5.0, 8.0, 11.0, 13.0])
colors_amr = plt.cm.viridis(np.linspace(0, 1, len(ages_gyr)))

for age_gyr, color in zip(ages_gyr, colors_amr):
    z_amr = Z_sun * (age_gyr / age_uni_gyr) * 0.3
    ax.scatter(age_gyr, z_amr / Z_sun, s=200, color=color, edgecolors="k", linewidth=1.0)

ages_interp = np.linspace(0, age_uni_gyr, 100)
z_interp = Z_sun * (ages_interp / age_uni_gyr) * 0.3
ax.plot(ages_interp, z_interp / Z_sun, "k--", lw=1.5, alpha=0.4, label="Age-Metallicity Relation")

ax.set_xlabel("Galaxy Age [Gyr]")
ax.set_ylabel(r"Metallicity (Z / Z$_\odot$)")
ax.legend(fontsize=10, frameon=False)
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.5, 14)
ax.set_ylim(-2.5, 0.5)

fig.tight_layout(rect=[0, 0, 1, 0.97])

# --- Add panels: Z(t) scenarios and resulting SEDs ---
# (from plot_chemical_evolution_ramp.py)
AGE_UNIVERSE_GYR = 13.8
C_AA_PER_S = 2.998e18

ssp = tengri.load_ssp()

# Scenario 1: Constant solar metallicity
spec_delta = tengri.Parameters(
    mean_sfh_type="dpl",
    sfh_dpl_tau_gyr=1.0,
    sfh_dpl_log_total_mass=10.0,
    dust_tau_diff=0.1,
    met_mode="delta",
    redshift=0.0,
)
model_delta = tengri.SEDModel(spec_delta, ssp)

# Scenario 2: Linear ramp from low to solar metallicity
spec_ramp = tengri.Parameters(
    mean_sfh_type="dpl",
    sfh_dpl_tau_gyr=1.0,
    sfh_dpl_log_total_mass=10.0,
    dust_tau_diff=0.1,
    met_mode="ramp",
    redshift=0.0,
)
model_ramp = tengri.SEDModel(spec_ramp, ssp)

# Scenario 3: Two-step metallicity
spec_twostep = tengri.Parameters(
    mean_sfh_type="dpl",
    sfh_dpl_tau_gyr=1.0,
    sfh_dpl_log_total_mass=10.0,
    dust_tau_diff=0.1,
    met_mode="two_step",
    redshift=0.0,
)
model_twostep = tengri.SEDModel(spec_twostep, ssp)

# Sample parameters
p_delta = dict(model_delta.spec.sample(jax.random.PRNGKey(42)))
p_delta["met_logzsol"] = 0.0

p_ramp = dict(model_ramp.spec.sample(jax.random.PRNGKey(43)))
p_ramp["met_logzsol_0"] = -1.0
p_ramp["met_logzsol_final"] = 0.0

p_twostep = dict(model_twostep.spec.sample(jax.random.PRNGKey(44)))
p_twostep["met_logzsol_old"] = -0.5
p_twostep["met_logzsol_young"] = 0.0
p_twostep["met_step_age_gyr"] = 8.0

pred_delta = model_delta.predict_rest_sed(p_delta)
pred_ramp = model_ramp.predict_rest_sed(p_ramp)
pred_twostep = model_twostep.predict_rest_sed(p_twostep)

wave_delta = np.asarray(pred_delta.wavelength)
sed_delta = np.asarray(pred_delta.sed)
wave_ramp = np.asarray(pred_ramp.wavelength)
sed_ramp = np.asarray(pred_ramp.sed)
wave_twostep = np.asarray(pred_twostep.wavelength)
sed_twostep = np.asarray(pred_twostep.sed)

# Z(t) extraction
ssp_lg_ages_gyr = np.asarray(ssp.ssp_lg_age_gyr)
ssp_ages_gyr = 10.0**ssp_lg_ages_gyr
ssp_ages_yr = ssp_ages_gyr * 1e9
lookback_time_gyr = AGE_UNIVERSE_GYR - ssp_ages_gyr

z_delta = np.full_like(lookback_time_gyr, 10.0 ** p_delta["met_logzsol"])

z_0 = 10.0 ** p_ramp["met_logzsol_0"]
z_final = 10.0 ** p_ramp["met_logzsol_final"]
age_max = ssp_ages_yr.max()
z_ramp = z_0 + (z_final - z_0) * (ssp_ages_yr / age_max)

step_age_gyr = p_twostep["met_step_age_gyr"]
step_age_yr = step_age_gyr * 1e9
z_old = 10.0 ** p_twostep["met_logzsol_old"]
z_young = 10.0 ** p_twostep["met_logzsol_young"]
z_twostep = np.where(ssp_ages_yr >= step_age_yr, z_old, z_young)

# Create new figure with Z(t) and SED panels
fig_met = plt.figure(figsize=(13, 6))
ax_zt = fig_met.add_subplot(121)
ax_sed = fig_met.add_subplot(122)

colors = {
    "delta": "#1f77b4",
    "ramp": "#ff7f0e",
    "twostep": "#2ca02c",
}

ax_zt.plot(lookback_time_gyr, z_delta, color=colors["delta"], lw=2.0, label="Constant Z (0.0 dex)")
ax_zt.plot(lookback_time_gyr, z_ramp, color=colors["ramp"], lw=2.0, label="Ramp: -1.0 → 0.0 dex")
ax_zt.plot(
    lookback_time_gyr, z_twostep, color=colors["twostep"], lw=2.0, label="Two-step (step at 8 Gyr)"
)
ax_zt.axhline(1.0, color="0.5", lw=0.8, ls="--", alpha=0.5)
ax_zt.text(13, 1.02, r"Solar $Z_\odot$", fontsize=9, alpha=0.6)
ax_zt.set(
    xlabel=r"Lookback time [Gyr]",
    ylabel=r"Metallicity $Z$ [$Z_\odot$]",
    xlim=(0, AGE_UNIVERSE_GYR),
    ylim=(0.05, 1.5),
)
ax_zt.legend(frameon=False, fontsize=9, loc="upper left")
ax_zt.grid(True, alpha=0.25, which="major")

ax_sed.loglog(wave_delta, sed_delta, color=colors["delta"], lw=2.0, label="Constant Z")
ax_sed.loglog(wave_ramp, sed_ramp, color=colors["ramp"], lw=2.0, label="Ramp")
ax_sed.loglog(wave_twostep, sed_twostep, color=colors["twostep"], lw=2.0, label="Two-step")
ax_sed.set(
    xlabel=r"Rest-frame wavelength [$\mathrm{\AA}$]",
    ylabel=r"$\nu L_\nu$ [erg/s]",
    xlim=(500, 1e5),
)
ax_sed.legend(frameon=False, fontsize=9, loc="upper right")
ax_sed.grid(True, alpha=0.25, which="both", axis="both")

fig_met.tight_layout()
plt.savefig("plot_chemical_evolution.png", dpi=150, bbox_inches="tight")